In [7]:
import pandas as pd
import os

src = "/home/sxi219/RAI-Account/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/data/graphs/"
cwe = "CWE-119"
cve = "CVE-2019-"

cves = [i for i in os.listdir(src+cwe) if cve in i]

print(cves)



['CVE-2019-1010208', 'CVE-2019-1010292', 'CVE-2019-1010295', 'CVE-2019-1010296', 'CVE-2019-1010297', 'CVE-2019-1010298', 'CVE-2019-1010305', 'CVE-2019-10131', 'CVE-2019-11222', 'CVE-2019-11360', 'CVE-2019-11411', 'CVE-2019-11412', 'CVE-2019-12951', 'CVE-2019-12981', 'CVE-2019-12982', 'CVE-2019-13298', 'CVE-2019-13300', 'CVE-2019-13304', 'CVE-2019-13305', 'CVE-2019-13306', 'CVE-2019-13307', 'CVE-2019-13308', 'CVE-2019-14323', 'CVE-2019-15296', 'CVE-2019-15785', 'CVE-2019-15937', 'CVE-2019-15938', 'CVE-2019-15945', 'CVE-2019-15946', 'CVE-2019-16058', 'CVE-2019-16347', 'CVE-2019-5824', 'CVE-2019-9578']


# IPAG Builder Experiment

In [7]:
from ipag_gin.graph.ipag_builder import IPAGBuilder
from ipag_gin.graph.build_language import LanguageBuilder

langs = set(df_code_lang.str.lower())
lang_map = LanguageBuilder(langs)
ipag = IPAGBuilder(source = df_code_after, language = df_code_lang.str.lower(), lang_map=lang_map.build())
ipag.build()
df_ipag = ipag.get_ipag_dataframe()


Building ASTs for all code snippets
Processed 100/1000 snippets...
Processed 200/1000 snippets...
Processed 300/1000 snippets...
Processed 400/1000 snippets...
Processed 500/1000 snippets...
Processed 600/1000 snippets...
Processed 700/1000 snippets...
Processed 800/1000 snippets...
Processed 900/1000 snippets...
Processed 1000/1000 snippets...
Finished building ASTs
Successful: 1000, Failed: 0, Total: 1000
IPAG Construction Complete
Total snippets: 1000
Average nodes per snippet: 261.3
Average edges per snippet: 260.3


## Building Node features for GIN training

In [8]:
from ipag_gin.graph.features_accelerated import BuildNodeFeaturesAccelerated
import torch

torch.cuda.empty_cache()

# builder = BuildNodeFeaturesAccelerated(device='cuda', batch_size=1280)

all_ipag_nodes = df_ipag['ipag_nodes'].tolist()
all_ipag_edges = df_ipag['ipag_edges'].tolist()

# features_batch = builder.process_ipag_batch(all_ipag_nodes, all_ipag_edges)

In [9]:
from ipag_gin.utils.worker import Worker

def local_worker(ipag_nodes, ipag_edges, cuda_device_idx, device, batch_size, save_path):
    worker = Worker(BuildNodeFeaturesAccelerated)
    builder = worker.set_args(cuda_device_idx = cuda_device_idx, device = device, batch_size = batch_size)
    results = builder.process_ipag_batch(ipag_nodes, ipag_edges)
    builder.save_features(results, save_path)

total_graphs = len(all_ipag_nodes)
mid = total_graphs // 2

chunk_0_nodes = all_ipag_nodes[:mid]
chunk_1_nodes = all_ipag_nodes[mid:]

chunk_0_edges = all_ipag_edges[:mid]
chunk_1_edges = all_ipag_edges[mid:]



In [10]:
from multiprocessing import Process


p0 = Process(target=local_worker, args=(chunk_0_nodes, chunk_0_edges, 0, 'cuda', 2560, '/mnt/vstor/courses/csds447/sxi219/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/data/processed/batch0.pkl'))
p1 = Process(target=local_worker, args=(chunk_1_nodes, chunk_1_edges, 1, 'cuda', 2560, '/mnt/vstor/courses/csds447/sxi219/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/data/processed/batch1.pkl'))

p0.start()
p1.start()
p0.join()
p1.join()

print("Finished! Features for both GPUs saved.")



Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BuildNodeFeaturesAccelerated initialized on device: cuda with batch size: 2560
BuildNodeFeaturesAccelerated initialized on device: cuda with batch size: 2560
Collected 107885 total nodes. Starting batched encoding...
Collected 153387 total nodes. Starting batched encoding...


Process Process-1:
Traceback (most recent call last):
  File "/usr/local/easybuild_allnodes/software/Python/3.11.3-GCCcore-12.3.0/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/local/easybuild_allnodes/software/Python/3.11.3-GCCcore-12.3.0/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/job.44047.markov2/ipykernel_2321793/3983666070.py", line 6, in local_worker
    results = builder.process_ipag_batch(ipag_nodes, ipag_edges)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/vstor/courses/csds447/sxi219/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/src/ipag_gin/graph/features_accelerated.py", line 393, in process_ipag_batch
    global_embedding_map = self.get_all_node_embeddings_batch(ipag_nodes_list)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/vstor/courses/csds447/sxi219/multi-stage-GN

KeyboardInterrupt: 

Process Process-2:
Traceback (most recent call last):
  File "/usr/local/easybuild_allnodes/software/Python/3.11.3-GCCcore-12.3.0/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/local/easybuild_allnodes/software/Python/3.11.3-GCCcore-12.3.0/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/job.44047.markov2/ipykernel_2321793/3983666070.py", line 6, in local_worker
    results = builder.process_ipag_batch(ipag_nodes, ipag_edges)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/vstor/courses/csds447/sxi219/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/src/ipag_gin/graph/features_accelerated.py", line 393, in process_ipag_batch
    global_embedding_map = self.get_all_node_embeddings_batch(ipag_nodes_list)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/vstor/courses/csds447/sxi219/multi-stage-GN

# GIN Classifier

## Preprocess the code

In [ ]:
from ipag_gin.graph.ipag_builder import IPAGBuilder
from ipag_gin.graph.build_language import LanguageBuilder


def process_single_code(code: str, language: str):
    """
    Process a single code sample and return its IPAG representation.
    
    Args:
        code: Source code string to process
        language: Programming language (e.g., 'c', 'cpp', 'python')
    
    Returns:
        dict: Dictionary containing 'ipag_nodes' and 'ipag_edges'
    """
    # Validate inputs
    if not code or not isinstance(code, str) or code.strip() == "":
        raise ValueError("Code must be a non-empty string")
    
    if not language or not isinstance(language, str):
        raise ValueError("Language must be a non-empty string")
    
    # Normalize language to lowercase
    language = language.lower()
    
    # Build language map for the single language
    lang_map = LanguageBuilder({language})
    
    # Build IPAG for single code sample
    ipag = IPAGBuilder(
        source=[code], 
        language=[language], 
        lang_map=lang_map.build()
    )
    ipag.build()
    
    # Get IPAG dataframe
    df_ipag = ipag.get_ipag_dataframe()
    
    # Extract nodes and edges for the single sample
    ipag_nodes = df_ipag['ipag_nodes'].iloc[0]
    ipag_edges = df_ipag['ipag_edges'].iloc[0]
    
    return {
        'ipag_nodes': ipag_nodes,
        'ipag_edges': ipag_edges
    }



## Building Features

In [ ]:
from transformers import RobertaTokenizer, RobertaModel
import torch
import numpy as np
from collections import Counter


def extract_ipag_features(ipag_nodes, ipag_edges, device=None, cuda_device_idx=0):
    """
    Extract comprehensive features from a single IPAG using GraphCodeBERT.
    
    Args:
        ipag_nodes (list): List of node dictionaries from IPAG
        ipag_edges (list): List of edge dictionaries from IPAG
        device (str or None): Torch device string; e.g., 'cuda', 'cpu'
        cuda_device_idx (int): CUDA device index for multi-GPU systems
    
    Returns:
        dict: Dictionary containing:
            - 'node_features': dict mapping node_id to feature dict
            - 'edge_features': list of edge feature dicts
            - 'graph_features': dict of graph-level features
    """
    # === Initialize Model ===
    tokenizer = RobertaTokenizer.from_pretrained("microsoft/graphcodebert-base")
    model = RobertaModel.from_pretrained("microsoft/graphcodebert-base")
    model.eval()
    
    # Device selection
    if device is None:
        if torch.cuda.is_available():
            device = torch.device(f'cuda:{cuda_device_idx}')
        else:
            device = torch.device('cpu')
    else:
        device = torch.device(device)
    
    model.to(device)
    
    # === Helper: Encode text ===
    def encode_text(text, max_length=64):
        sanitized = text if isinstance(text, str) else ""
        inputs = tokenizer(
            sanitized,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            embedding = outputs.last_hidden_state[0, 0, :].cpu().numpy()
        
        return embedding
    
    # === Helper: Get node type encoding ===
    def get_node_type_encoding(node_type):
        type_map = {'TOKEN': 0, 'DECLARATION': 1, 'PROPERTY': 2}
        encoding = np.zeros(3, dtype=np.float32)
        
        if node_type in type_map:
            encoding[type_map[node_type]] = 1.0
        else:
            encoding[2] = 1.0  # Default to PROPERTY
        
        return encoding
    
    # === Helper: Compute structural features ===
    def compute_structural_features(node_ids, edges):
        in_counts = Counter()
        out_counts = Counter()
        
        for edge in edges:
            out_counts[edge['source']] += 1
            in_counts[edge['target']] += 1
        
        features = {}
        for node_id in node_ids:
            in_degree = in_counts.get(node_id, 0)
            out_degree = out_counts.get(node_id, 0)
            
            features[node_id] = {
                'degree': in_degree + out_degree,
                'in_degree': in_degree,
                'out_degree': out_degree,
                'is_leaf': 1 if out_degree == 0 else 0,
                'is_root': 1 if in_degree == 0 else 0
            }
        
        return features
    
    # === Extract Node Features ===
    if not ipag_nodes:
        return {
            'node_features': {},
            'edge_features': [],
            'graph_features': {}
        }
    
    # Get embeddings for all nodes
    node_embeddings = {}
    for node in ipag_nodes:
        node_id = node['id']
        label = node.get('label', '')
        node_type = node.get('type', 'PROPERTY')
        original_type = node.get('original_type', '')
        
        # Create text representation
        if label:
            text = f"{node_type}: {label}"
        else:
            text = f"{node_type}: {original_type}"
        
        # Get embedding
        embedding = encode_text(text)
        node_embeddings[node_id] = embedding
    
    # Get structural features
    node_ids = [node['id'] for node in ipag_nodes]
    structural_features = compute_structural_features(node_ids, ipag_edges)
    
    # Combine all node features
    node_features = {}
    for node in ipag_nodes:
        node_id = node['id']
        node_type = node.get('type', 'PROPERTY')
        
        embedding = node_embeddings.get(node_id, np.zeros(768, dtype=np.float32))
        type_encoding = get_node_type_encoding(node_type)
        structural = structural_features.get(node_id, {
            'degree': 0, 'in_degree': 0, 'out_degree': 0,
            'is_leaf': 0, 'is_root': 0
        })
        
        structural_vector = np.array([
            structural['degree'],
            structural['in_degree'],
            structural['out_degree'],
            structural['is_leaf'],
            structural['is_root']
        ], dtype=np.float32)
        
        # Combine all features
        combined = np.concatenate([
            embedding,
            type_encoding,
            structural_vector
        ])
        
        node_features[node_id] = {
            'embedding': embedding,
            'type_encoding': type_encoding,
            'structural': structural_vector,
            'combined': combined,
            'metadata': {
                'type': node_type,
                'label': node.get('label', ''),
                'original_type': node.get('original_type', ''),
                **structural
            }
        }
    
    # === Extract Edge Features ===
    edge_features = []
    
    for edge in ipag_edges:
        source_id = edge['source']
        target_id = edge['target']
        
        source_feat = node_features.get(source_id)
        target_feat = node_features.get(target_id)
        
        if not source_feat or not target_feat:
            continue
        
        source_emb = source_feat['embedding']
        target_emb = target_feat['embedding']
        
        # Edge features
        concatenated = np.concatenate([source_emb, target_emb])
        element_wise_product = source_emb * target_emb
        
        # Cosine similarity
        norm_source = np.linalg.norm(source_emb)
        norm_target = np.linalg.norm(target_emb)
        cosine_sim = np.dot(source_emb, target_emb) / (norm_source * norm_target + 1e-8)
        
        edge_feature = {
            'source': source_id,
            'target': target_id,
            'type': edge.get('type', 'CHILD'),
            'source_embedding': source_emb,
            'target_embedding': target_emb,
            'concatenated': concatenated,
            'element_wise_product': element_wise_product,
            'cosine_similarity': cosine_sim
        }
        edge_features.append(edge_feature)
    
    # === Extract Graph-Level Features ===
    num_nodes = len(ipag_nodes)
    num_edges = len(ipag_edges)
    
    node_type_counts = Counter(node.get('type') for node in ipag_nodes)
    
    # Graph statistics
    if node_features:
        all_embeddings = np.array([feat['embedding'] for feat in node_features.values()])
        avg_embedding = np.mean(all_embeddings, axis=0)
        
        degrees = np.array([feat['metadata']['degree'] for feat in node_features.values()])
        avg_degree = np.mean(degrees)
        max_degree = np.max(degrees)
        
        num_leaves = sum(1 for feat in node_features.values() if feat['metadata']['is_leaf'])
        num_roots = sum(1 for feat in node_features.values() if feat['metadata']['is_root'])
    else:
        avg_embedding = np.zeros(768)
        avg_degree = 0
        max_degree = 0
        num_leaves = 0
        num_roots = 0
    
    graph_features = {
        'num_nodes': num_nodes,
        'num_edges': num_edges,
        'num_tokens': node_type_counts.get('TOKEN', 0),
        'num_declarations': node_type_counts.get('DECLARATION', 0),
        'num_properties': node_type_counts.get('PROPERTY', 0),
        'avg_degree': float(avg_degree),
        'max_degree': int(max_degree),
        'num_leaves': num_leaves,
        'num_roots': num_roots,
        'avg_embedding': avg_embedding,
        'graph_density': num_edges / (num_nodes * (num_nodes - 1)) if num_nodes > 1 else 0
    }
    
    return {
        'node_features': node_features,
        'edge_features': edge_features,
        'graph_features': graph_features
    }



## Passing the Feature to model to predict

In [4]:
import json
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GINConv, global_mean_pool, global_add_pool

def load_features_from_pkl(pkl_path: str):
    """
    Load features from a pickle file.
    
    Args:
        pkl_path (str): Path to the .pkl file
    
    Returns:
        dict: The features dictionary
    """
    with open(pkl_path, 'rb') as f:
        features = pickle.load(f)
    return features

config_path = "/home/sxi219/RAI-Account/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/outputs/experiments/exp8_tiny_heavy_reg/config.json"
model_path = "/home/sxi219/RAI-Account/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/outputs/experiments/exp8_tiny_heavy_reg/final_model.pth"
code_pkl_path = "code_features.pkl"

config = json.load(config_path)
code_features = load_features_from_pkl(code_pkl_path)

from ipag_gin.model.gin_classifier import GINVulnerabilityClassifier
import torch

model = GINVulnerabilityClassifier(
            input_dim=self.config['input_dim'],
            hidden_dims=self.config['hidden_dims'],
            output_dim=self.config['output_dim'],
            dropout=self.config['dropout'],
            pooling=self.config['pooling']
        )
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(model_path, map_location=dev)
model = model.to(self.device)
model.eval()

node_features_dict = code_features['node_features']
edge_features_list = code_features['edge_features']

node_ids = sorted(node_features_dict.keys())
node_embeddings = np.array([node_features_dict[nid]['combined'] for nid in node_ids])
x = torch.tensor(node_embeddings, dtype=torch.float32)

# Build edge index
node_id_to_idx = {nid: i for i, nid in enumerate(node_ids)}
edge_index_list = []

for edge_feat in edge_features_list:
    src_id = edge_feat['source']
    tgt_id = edge_feat['target']
    
    if src_id in node_id_to_idx and tgt_id in node_id_to_idx:
        src_idx = node_id_to_idx[src_id]
        tgt_idx = node_id_to_idx[tgt_id]
        edge_index_list.append([src_idx, tgt_idx])

if edge_index_list:
    edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
else:
    edge_index = torch.zeros((2, 0), dtype=torch.long)

# Create PyG Data object
data = Data(
    x=x,
    edge_index=edge_index,
    y=torch.tensor(label if label is not None else -1, dtype=torch.long),
    num_nodes=len(node_ids)
)

data = data.to(dev)
with torch.no_grad():
    logits = model(data)
    probs = F.softmax(logits, dim=1)
    pred = logits.argmax(dim=1).item()
    confidence = probs[0, pred].item()

result = {
    'file': str(code_pkl_path),
    'prediction': pred,
    'prediction_label': 'Vulnerable' if pred == 1 else 'Non-Vulnerable',
    'confidence': confidence
}

AttributeError: 'str' object has no attribute 'read'

In [6]:
import json
import pickle
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data


def predict_vulnerability(config_path: str, model_path: str, code_pkl_path: str):
    """
    Predict vulnerability for a code sample using a trained GIN model.
    
    Args:
        config_path (str): Path to the model configuration JSON file
        model_path (str): Path to the trained model checkpoint (.pth file)
        code_pkl_path (str): Path to the code features pickle file
    
    Returns:
        dict: Dictionary containing:
            - file: Path to the input code features file
            - prediction: Predicted class (0 or 1)
            - prediction_label: Human-readable prediction label
            - confidence: Confidence score for the prediction
    """
    # Load configuration
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    # Load code features from pickle
    with open(code_pkl_path, 'rb') as f:
        code_features = pickle.load(f)
    
    # Import model class (adjust import path as needed)
    from ipag_gin.model.gin_classifier import GINVulnerabilityClassifier
    
    # Initialize model
    model = GINVulnerabilityClassifier(
        input_dim=config['input_dim'],
        hidden_dims=config['hidden_dims'],
        output_dim=config['output_dim'],
        dropout=config['dropout'],
        pooling=config['pooling']
    )
    
    # Load model checkpoint
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(model_path, map_location=device)
    
    # Extract model state dict from checkpoint
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    
    model = model.to(device)
    model.eval()
    
    # Extract node and edge features
    node_features_dict = code_features['node_features']
    edge_features_list = code_features['edge_features']
    
    # Build node feature matrix
    node_ids = sorted(node_features_dict.keys())
    node_embeddings = np.array([node_features_dict[nid]['combined'] for nid in node_ids])
    x = torch.tensor(node_embeddings, dtype=torch.float32)
    
    # Build edge index
    node_id_to_idx = {nid: i for i, nid in enumerate(node_ids)}
    edge_index_list = []
    
    for edge_feat in edge_features_list:
        src_id = edge_feat['source']
        tgt_id = edge_feat['target']
        
        if src_id in node_id_to_idx and tgt_id in node_id_to_idx:
            src_idx = node_id_to_idx[src_id]
            tgt_idx = node_id_to_idx[tgt_id]
            edge_index_list.append([src_idx, tgt_idx])
    
    if edge_index_list:
        edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    
    # Get label if available (default to -1 if not present)
    label = code_features.get('label', -1)
    
    # Create PyTorch Geometric Data object
    data = Data(
        x=x,
        edge_index=edge_index,
        y=torch.tensor(label, dtype=torch.long),
        num_nodes=len(node_ids)
    )
    
    # Move data to device and perform inference
    data = data.to(device)
    with torch.no_grad():
        logits = model(data)
        probs = F.softmax(logits, dim=1)
        pred = logits.argmax(dim=1).item()
        confidence = probs[0, pred].item()
    
    # Prepare result
    result = {
        'file': str(code_pkl_path),
        'prediction': pred,
        'prediction_label': 'Vulnerable' if pred == 1 else 'Non-Vulnerable',
        'confidence': confidence
    }
    
    return result


# Example usage:
if __name__ == "__main__":
    config_path = "/home/sxi219/RAI-Account/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/outputs/experiments/exp8_tiny_heavy_reg/config.json"
    model_path = "/home/sxi219/RAI-Account/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/outputs/experiments/exp8_tiny_heavy_reg/final_model.pth"
    code_pkl_path = "code_features.pkl"
    
    result = predict_vulnerability(config_path, model_path, code_pkl_path)
    print(json.dumps(result, indent=2))

/tmp/job.44631.markov2/ipykernel_2294165/595745230.py:47: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


{
  "file": "code_features.pkl",
  "prediction": 1,
  "prediction_label": "Vulnerable",
  "confidence": 0.8289250731468201
}
